# 02 - Synthetic Data Generation

This notebook generates the synthetic dataset for the Identity Fraud Detection project.

**What this notebook does:**
1. Imports the data generator module
2. Creates a 5,000-row prototype dataset
3. Explores the generated data
4. Validates and saves outputs

**Output files:**
- `data/raw/applications_prototype.parquet`
- `data/raw/applications_prototype.csv`
- `data/interim/generation_metadata.csv`

## Design Philosophy Summary

This generator follows a specific design philosophy (see `reports/dataset_design_philosophy.md`):

**Core Principles:**
1. **Believability** - Each row represents a plausible application, not random values
2. **Pattern Diversity** - Five distinct fraud archetypes with different signal patterns
3. **Ambiguity by Design** - Hard/borderline cases test model boundaries
4. **Controlled Randomness** - Reproducible via random seeds

**Fraud Archetypes Generated:**
- `legitimate_clean` (70%) - Standard clean applications
- `legitimate_noisy` (18%) - Legitimate but with ambiguity
- `synthetic_identity` (5%) - Fabricated/stitched identities
- `true_name_fraud` (4%) - Real identity misused
- `coordinated_attack` (3%) - Cluster-based attacks

**Key Design Decisions:**
- Fraud arises from realistic mechanisms, not random noise
- Difficulty levels control signal intensity (easy/medium/hard)
- Attack pools enable device reuse patterns
- Text templates create semantic consistency/inconsistency patterns

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path so we can import our modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import our data generator
from src.data_generation import (
    create_dataset,
    validate_dataset,
    save_outputs,
    print_summary,
    load_dictionaries
)

print("Imports successful!")
print(f"Project root: {project_root}")

## 2. Verify Dictionaries are Loaded

In [ ]:
# Quick check that all dictionary files are available
dictionaries = load_dictionaries()

print("Loaded dictionaries:")
for name, df in dictionaries.items():
    print(f"  {name}: {len(df)} rows")

## 3. Generate the Prototype Dataset

We'll generate 5,000 rows for the initial prototype.

In [ ]:
# Generate the dataset
# This may take a few seconds
df = create_dataset(n_rows=5000, seed=42)

print(f"\nGenerated {len(df)} rows")
print(f"Columns: {len(df.columns)}")

## 4. Explore the Dataset

In [ ]:
# View first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Check column names and types
print("Column info:")
df.info()

In [ ]:
# Fraud label distribution
print("Fraud Label Distribution:")
print(df["fraud_label"].value_counts())
print(f"\nFraud rate: {df['fraud_label'].mean() * 100:.2f}%")

In [ ]:
# Fraud type distribution
print("Fraud Type Distribution:")
fraud_type_counts = df["fraud_type"].value_counts()
for ft, count in fraud_type_counts.items():
    pct = count / len(df) * 100
    print(f"  {ft}: {count} ({pct:.1f}%)")

In [ ]:
# Difficulty level distribution
print("Difficulty Level Distribution:")
difficulty_counts = df["difficulty_level"].value_counts()
for diff, count in difficulty_counts.items():
    pct = count / len(df) * 100
    print(f"  {diff}: {count} ({pct:.1f}%)")

In [ ]:
# Difficulty by fraud type (cross-tabulation)
print("Difficulty by Fraud Type:")
pd.crosstab(df["fraud_type"], df["difficulty_level"], margins=True)

## 5. Examine Text Fields

Let's look at sample text fields to verify they look realistic.

In [ ]:
# Sample verification notes
print("Sample Verification Notes:")
print("=" * 60)

# Show 2 legitimate and 2 fraud examples
legit_samples = df[df["fraud_label"] == 0].sample(2, random_state=42)
fraud_samples = df[df["fraud_label"] == 1].sample(2, random_state=42)

print("\nLegitimate examples:")
for _, row in legit_samples.iterrows():
    print(f"  [{row['fraud_type']}] {row['verification_note']}")

print("\nFraud examples:")
for _, row in fraud_samples.iterrows():
    print(f"  [{row['fraud_type']}] {row['verification_note']}")

In [ ]:
# Sample OCR text
print("Sample OCR Document Text:")
print("=" * 60)

print("\nLegitimate examples:")
for _, row in legit_samples.iterrows():
    print(f"  [{row['fraud_type']}] {row['ocr_document_text']}")

print("\nFraud examples:")
for _, row in fraud_samples.iterrows():
    print(f"  [{row['fraud_type']}] {row['ocr_document_text']}")

In [ ]:
# View 5 complete sample rows with key fields
print("5 Sample Rows (Key Fields):")
sample_cols = [
    "application_id", "claimed_first_name", "claimed_last_name",
    "email", "fraud_type", "difficulty_level", "fraud_label"
]
df[sample_cols].sample(5, random_state=42)

## 6. Check Key Signal Distributions

In [ ]:
# Compare key signals between fraud and legitimate
print("Signal Comparison: Fraud vs Legitimate")
print("=" * 60)

signals = [
    "num_prev_apps_same_device_7d",
    "num_prev_apps_same_phone_30d",
    "num_prev_apps_same_address_30d",
    "months_at_address",
    "months_at_employer",
    "thin_file_flag",
    "zip_ip_distance_proxy",
    "name_email_match_score",
    "is_free_email_domain",
]

for signal in signals:
    legit_mean = df[df["fraud_label"] == 0][signal].mean()
    fraud_mean = df[df["fraud_label"] == 1][signal].mean()
    print(f"{signal}:")
    print(f"  Legitimate mean: {legit_mean:.3f}")
    print(f"  Fraud mean:      {fraud_mean:.3f}")
    print()

In [ ]:
# Generated signal score distribution
print("Generated Signal Score by Fraud Type:")
df.groupby("fraud_type")["generated_signal_score"].describe().round(3)

## 7. Validate the Dataset

In [ ]:
# Run validation checks
validate_dataset(df)
print("\nAll validation checks passed!")

## 8. Save Outputs

In [ ]:
# Save the dataset
output_paths = save_outputs(df, output_name="applications_prototype")

print("\nSaved files:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")

## 9. Final Summary

In [ ]:
# Print full summary
print_summary(df)

In [ ]:
# Quick stats for the report
print("\nQuick Stats for Report:")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Fraud rate: {df['fraud_label'].mean() * 100:.1f}%")
print(f"Date range: {df['application_date'].min()} to {df['application_date'].max()}")
print(f"Unique months: {df['application_month'].nunique()}")

## Done!

The prototype dataset has been generated and saved. Next steps:
- Review the data in `data/raw/applications_prototype.csv`
- Proceed to Phase 5 for exploratory data analysis
- Later: Build the Stage 1 structured fraud model